# PROJECT 1: WORLD STOCK PRICE ANALYSIS & PREDICTION - NGUYEN TUNG SON 
## DMP + EDA + ML + Streamlit Dashboard

**Mục tiêu:**
- Làm sạch dữ liệu giá cổ phiếu thế giới
- Phân tích khám phá (EDA)
- Feature engineering
- Huấn luyện mô hình Linear Regression dự đoán giá đóng cửa

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('World-Stock-Prices-Dataset.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.tail()

In [ ]:
print('Columns:', df.columns.tolist())
df.info()

## 3. Statistical Summary

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## 4. Data Cleaning - Column Name Standardization

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
print(df.columns.tolist())

## 5. Missing Value Analysis

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nMissing value percentage:')
print((df.isnull().sum() / len(df)) * 100)

In [ ]:
# Cột capital_gains thiếu gần 100% giá trị -> nên drop
if 'capital_gains' in df.columns:
    missing_pct = df['capital_gains'].isnull().sum() / len(df) * 100
    print(f'capital_gains missing: {missing_pct:.2f}% -> dropping column')
    df = df.drop(columns=['capital_gains'])

print('Remaining columns:', df.columns.tolist())

## 6. Handling Missing Values

In [ ]:
# Numerical Columns - dùng median (bền vững với giá cổ phiếu biến động mạnh)
num_cols = df.select_dtypes(include=np.number).columns
print('Numerical columns:', list(num_cols))

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
# Categorical Columns - dùng mode
cat_cols = df.select_dtypes(include='object').columns
print('Categorical columns:', list(cat_cols))

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('\nMissing values sau xử lý:')
print(df.isnull().sum())

## 7. Duplicate Value Check

In [ ]:
print('Số dòng trùng lặp:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('Shape sau khi drop duplicates:', df.shape)

## 8. Univariate Analysis (EDA) - Distribution of Numerical Features

In [ ]:
# Để code chạy nhanh, ta chỉ vẽ 4 cột quan trọng
main_price_cols = ['open', 'high', 'low', 'close', 'volume']
main_price_cols = [c for c in main_price_cols if c in df.columns]

for col in main_price_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True, bins=50)
    plt.title(f'{col} Distribution')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

## 9. Date Distribution (nếu có cột Date)

In [ ]:
if 'date' in df.columns:
    # utc=True vì dataset có timezone kèm theo
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
    df = df.dropna(subset=['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    
    plt.figure(figsize=(12, 5))
    sns.countplot(x='year', data=df)
    plt.xticks(rotation=90)
    plt.title('Year Distribution of Records')
    plt.xlabel('Year')
    plt.ylabel('Count')
    plt.show()

print('Date range:', df['date'].min(), 'to', df['date'].max())

## 10. Bivariate Analysis

In [ ]:
# Lấy sample 5000 dòng để scatter plot rõ hơn (dataset lớn 300k+ dòng)
sample_df = df.sample(n=5000, random_state=42)

# Open vs Close
plt.figure(figsize=(8, 5))
sns.scatterplot(x='open', y='close', data=sample_df, alpha=0.5)
plt.title('Open vs Close Price')
plt.show()

In [ ]:
# High vs Low
plt.figure(figsize=(8, 5))
sns.scatterplot(x='high', y='low', data=sample_df, alpha=0.5, color='green')
plt.title('High vs Low Price')
plt.show()

In [ ]:
# Volume vs Close
plt.figure(figsize=(8, 5))
sns.scatterplot(x='volume', y='close', data=sample_df, alpha=0.5, color='red')
plt.title('Volume vs Close Price')
plt.xscale('log')  # log scale cho volume
plt.show()

## 11. Multivariate Analysis - Correlation Heatmap

In [ ]:
# Lấy các cột số chính
numerical_cols = df.select_dtypes(include=np.number).columns

plt.figure(figsize=(10, 7))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Pairplot với 5000 sample cho nhanh
key_cols = ['open', 'high', 'low', 'close', 'volume']
available_cols = [col for col in key_cols if col in df.columns]

sns.pairplot(sample_df[available_cols])
plt.suptitle('Pairplot of Key Stock Features', y=1.02)
plt.show()

## 12. Outlier Detection (Boxplot)

In [ ]:
for col in ['open', 'high', 'low', 'close', 'volume']:
    if col in df.columns:
        plt.figure(figsize=(7, 3))
        sns.boxplot(x=df[col])
        plt.title(f'{col} - Outlier Detection')
        plt.show()

## 13. Feature Engineering

In [ ]:
# Sắp xếp theo Ticker và Date trước khi tính daily_return & moving average
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

# Daily Return tính theo từng ticker
if 'close' in df.columns and 'ticker' in df.columns:
    df['daily_return'] = df.groupby('ticker')['close'].pct_change()

# Moving Averages 20 & 50 ngày theo từng ticker
if 'close' in df.columns and 'ticker' in df.columns:
    df['ma_20'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
    df['ma_50'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(50).mean())

df[['ticker', 'date', 'close', 'daily_return', 'ma_20', 'ma_50']].head(60)

In [ ]:
# Các giá trị NaN ở daily_return, ma_20, ma_50 (ở đầu mỗi ticker) - fill 0 hoặc median
df['daily_return'] = df['daily_return'].fillna(0)
df['ma_20'] = df['ma_20'].fillna(df['close'])
df['ma_50'] = df['ma_50'].fillna(df['close'])
print('Missing values sau feature engineering:')
print(df[['daily_return', 'ma_20', 'ma_50']].isnull().sum())

In [ ]:
# Visualize Close Price + MA20 + MA50 cho 1 ticker ví dụ (Apple)
example_ticker = 'AAPL'
if example_ticker in df['ticker'].unique():
    plot_df = df[df['ticker'] == example_ticker].sort_values('date')
    plt.figure(figsize=(14, 5))
    plt.plot(plot_df['date'], plot_df['close'], label='Close Price', alpha=0.6)
    plt.plot(plot_df['date'], plot_df['ma_20'], label='MA 20-day', color='orange')
    plt.plot(plot_df['date'], plot_df['ma_50'], label='MA 50-day', color='red')
    plt.title(f'{example_ticker} - Close Price with Moving Averages')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.show()

## 14. Feature Encoding & Scaling (for ML)

In [ ]:
# Chuẩn bị data cho model Linear Regression
# Không encode tất cả (Brand_Name có 62 giá trị, Ticker 62 giá trị -> sẽ tạo quá nhiều cột)
# Chỉ dùng: open, high, low, volume, ma_20, ma_50, daily_return để dự đoán close

feature_cols = ['open', 'high', 'low', 'volume', 'ma_20', 'ma_50', 'daily_return']
feature_cols = [c for c in feature_cols if c in df.columns]
target_col = 'close'

X = df[feature_cols].copy()
y = df[target_col].copy()

print('Features:', feature_cols)
print('X shape:', X.shape, '| y shape:', y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=X.index)
X_scaled.head()

## 15. Train Linear Regression Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, '| Test:', X_test.shape)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f'R² Score : {r2_score(y_test, y_pred):.4f}')
print(f'MAE       : {mean_absolute_error(y_test, y_pred):.4f}')
print(f'RMSE      : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')

In [ ]:
# Visualize Predicted vs Actual
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual Close Price')
plt.ylabel('Predicted Close Price')
plt.title('Actual vs Predicted Close Prices (Linear Regression)')
plt.show()

In [ ]:
# Hệ số mô hình
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print('Intercept:', model.intercept_)
coef_df

## 16. Save Cleaned Dataset

In [ ]:
df.to_csv('World_Stock_Prices_Cleaned.csv', index=False)
print('✅ Saved: World_Stock_Prices_Cleaned.csv')
print('Shape:', df.shape)

## 17. Conclusion

**Kết quả đạt được:**

1. Dataset giá cổ phiếu thế giới đã được làm sạch: xử lý missing values, bỏ cột capital_gains (>99% thiếu), drop duplicates.
2. EDA cho thấy **tương quan rất mạnh** giữa các cột Open / High / Low / Close (thường > 0.99) — nghĩa là giá trong cùng 1 ngày dao động không quá lớn.
3. Feature Engineering đã bổ sung: `year`, `month`, `day`, `daily_return`, `ma_20`, `ma_50`.
4. Mô hình Linear Regression đạt R² rất cao (≈ 0.999) vì open/high/low đã chứa gần hết thông tin của close.
5. Dataset sạch đã được lưu vào `World_Stock_Prices_Cleaned.csv` để deploy trên Streamlit dashboard.